# $ \text{Т.5} $

$$ \text{Случайная величина распределена равномерно на отрезке } [\theta, 2\theta] \text{.}$$

In [23]:
import numpy as np

## $ \text{f) Сгенерировать выборку объёма } n = 100 \\ \text{ для некоторого значения параметра } \theta \text{.} \\ \text{ Вычислите указанные выше (по задаче)} \\ \text{ доверительные интервалы для } \beta = 0.95 \text{.}$

In [24]:
def p(x, theta):
  return 1 / theta if (2 * theta >= x >= theta) else 0


def F(x, theta):
  return x / theta - 1 if (2 * theta >= x >= theta) else 0


def InverseF(y, theta):
  return theta * (y + 1)

N = 100
beta = 0.95
theta = 10000

confidence_intervals: dict[str, int] = {}

In [25]:
rng = np.random.default_rng()
random_numbers = rng.random(size=N)

sample = np.array([InverseF(y, theta) for y in random_numbers])

In [26]:
print("Сгенерированная выборка из равномерного распределения U[theta, 2*theta]")
print(f"Параметр theta = {theta}")
print(f"Объём выборки N = {N}\n")

print("\nПолная выборка:")
print(np.array2string(sample, precision=2, threshold=np.inf))

Сгенерированная выборка из равномерного распределения U[theta, 2*theta]
Параметр theta = 10000
Объём выборки N = 100


Полная выборка:
[17741.07 17275.76 10527.15 17348.84 17909.74 18074.06 19079.8  13904.11
 12381.75 12203.78 19262.85 10243.25 14033.69 19255.18 11802.16 13981.65
 19010.59 15852.34 19707.07 14576.85 13733.6  15096.3  12723.26 10895.16
 12072.29 19272.43 18254.39 11465.5  10228.35 13076.95 10782.91 10933.43
 18447.62 15854.4  15758.47 19951.94 16308.91 14415.66 14501.25 19804.54
 16240.56 13630.24 17322.93 14305.94 13684.93 19334.22 18745.01 13635.73
 16454.97 15644.17 12685.74 18441.04 11517.51 10814.28 11555.54 15106.87
 13414.41 17980.86 10055.19 14955.74 10242.8  10147.27 15839.57 18443.29
 19176.81 17059.17 10292.05 15831.14 11636.05 15733.79 13847.85 15949.58
 13602.91 16755.58 14536.23 19585.09 14302.18 17807.32 19737.54 11108.48
 14299.35 15953.   16092.78 19860.27 15360.19 10094.92 12863.42 14115.54
 11151.69 13744.43 13756.08 18140.18 18570.99 18237.9  12871.9

$$ \text{Точный доверительный интервал}: $$

$$
P(\frac{x_{(n)}}{1 + \sqrt[n]{\frac{1 + \beta}{2}}} < \theta < \frac{x_{(n)}}{1 + \sqrt[n]{\frac{1 - \beta}{2}}})=\beta
$$

In [27]:
lower_bound = np.max(sample) / (1 + ((1 + beta) / 2) ** (1 / N))
upper_bound = np.max(sample) / (1 + ((1 - beta) / 2) ** (1 / N))

# Красивый вывод с использованием стандартного print
print("════════════════════════════════════════")
print("   Точный доверительный интервал")
print("════════════════════════════════════════")
print(f"   {lower_bound:.6f}  <  θ  <  {upper_bound:.6f}")
print(f"   Длина интервала: {upper_bound - lower_bound:.6f}")
print("════════════════════════════════════════\n")

confidence_intervals["Точный доверительный интервал"] = upper_bound - lower_bound

════════════════════════════════════════
   Точный доверительный интервал
════════════════════════════════════════
   9977.235018  <  θ  <  10159.952100
   Длина интервала: 182.717083
════════════════════════════════════════



$$ \text{Асимптотический доверительный интервал (ОММ):} $$

$$ P \left( \frac{2}{3} \overline{x} - \frac{3.92 \sqrt{S^2(n-1)}}{3n} < \theta < \frac{2}{3} \overline{x} + \frac{3.92 \sqrt{S^2(n-1)}}{3n} \right) = \beta $$

In [28]:
S = (np.sum([(x - np.mean(sample)) ** 2 for x in sample])) / (N - 1)

lower_bound = 2 * np.mean(sample) / 3 - 3.92 * np.sqrt(S * (N - 1)) / (3 * N)
upper_bound = 2 * np.mean(sample) / 3 + 3.92 * np.sqrt(S * (N - 1)) / (3 * N)

print("════════════════════════════════════════════════════════")
print("   Асимптотический доверительный интервал (О.М.М.)")
print("════════════════════════════════════════════════════════")
print(f"   {lower_bound:.6f}  <  θ  <  {upper_bound:.6f}")
print(f"   Длина интервала: {upper_bound - lower_bound:.6f}")
print("════════════════════════════════════════════════════════\n")

confidence_intervals["Асимптотический доверительный интервал (О.М.М.)"] = upper_bound - lower_bound

════════════════════════════════════════════════════════
   Асимптотический доверительный интервал (О.М.М.)
════════════════════════════════════════════════════════
   9654.958074  <  θ  <  10426.199568
   Длина интервала: 771.241494
════════════════════════════════════════════════════════



## $ \text{g) Численно постройте bootstrap'овский} \\ \text{ доверительный интервал.} $

In [29]:
bootstrap_iteration = 1000

theta_OMP = (N + 1) * np.max(sample) / (2 * N + 1)

In [30]:
bootstrap_delta: list[float] = []

for _ in range(bootstrap_iteration):
  bootstrap_delta.append((N + 1) * np.max(np.random.choice(sample, size=N)) /
                         (2 * N + 1) - theta_OMP)

In [31]:
lower_bound = theta_OMP - sorted(bootstrap_delta)[int((1 + beta) * bootstrap_iteration / 2)]
upper_bound = theta_OMP - sorted(bootstrap_delta)[int((1 - beta) * bootstrap_iteration / 2)]

print("═══════════════════════════════════════════════")
print("   bootstrap'овский доверительный интервал")
print("═══════════════════════════════════════════════")
print(f"   {lower_bound:.6f}  <  θ  <  {upper_bound:.6f}")
print(f"   Длина интервала: {upper_bound - lower_bound:.6f}")
print("═══════════════════════════════════════════════\n")

confidence_intervals["bootstrap'овский доверительный интервал"] = upper_bound - lower_bound

═══════════════════════════════════════════════
   bootstrap'овский доверительный интервал
═══════════════════════════════════════════════
   10025.603871  <  θ  <  10133.337987
   Длина интервала: 107.734116
═══════════════════════════════════════════════



## $ \text{h)  Сравнить все интервалы} $

In [32]:
confidence_intervals = sorted(confidence_intervals.items(), key=lambda item: item[1])

print("═══════════════════════════════════════════════")
print("   Доверительные интервалы (в порядке улучшения)")
print("═══════════════════════════════════════════════")
for interval_name, interval_value in confidence_intervals:
    print(f"   • {interval_name}  (l = {interval_value:.2f})")
print("═══════════════════════════════════════════════")

═══════════════════════════════════════════════
   Доверительные интервалы (в порядке улучшения)
═══════════════════════════════════════════════
   • bootstrap'овский доверительный интервал  (l = 107.73)
   • Точный доверительный интервал  (l = 182.72)
   • Асимптотический доверительный интервал (О.М.М.)  (l = 771.24)
═══════════════════════════════════════════════
